# Employee Performance Analytics — Exploratory Data Analysis

This notebook performs a comprehensive exploratory analysis of the employee performance dataset.
We examine employee demographics, performance evaluation trends, training effectiveness,
and promotion patterns to uncover actionable insights for HR decision-making.

## 1. Setup & Data Loading

In [ ]:
import sys
import os

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns

# Allow imports from src/
sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))

from data.generator import PerformanceDataGenerator

# Plotting defaults
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

print("Libraries loaded successfully.")

In [ ]:
# Generate synthetic dataset with default parameters
generator = PerformanceDataGenerator()
data = generator.generate()

# Unpack the returned tables
employees = data["employees"]
evaluations = data["evaluations"]
training = data["training"]
promotions = data["promotions"]

print(f"Generated {len(employees)} employees, {len(evaluations)} evaluations, "
      f"{len(training)} training records, {len(promotions)} promotions.")

## 2. Dataset Overview

In [ ]:
# Shape and dtypes for each table
tables = {
    "employees": employees,
    "evaluations": evaluations,
    "training": training,
    "promotions": promotions,
}

for name, df in tables.items():
    print(f"\n{'=' * 50}")
    print(f"Table: {name}  |  Shape: {df.shape}")
    print(f"{'=' * 50}")
    print(df.dtypes)
    print(f"\nNull counts:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

In [ ]:
# First 5 rows of employees
employees.head()

In [ ]:
# First 5 rows of evaluations
evaluations.head()

## 3. Employee Demographics

In [ ]:
# Department distribution
dept_counts = employees["department"].value_counts().reset_index()
dept_counts.columns = ["department", "count"]

fig = px.bar(
    dept_counts,
    x="department",
    y="count",
    color="department",
    title="Employee Count by Department",
    labels={"count": "Number of Employees", "department": "Department"},
)
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
# Age distribution
fig = px.histogram(
    employees,
    x="age",
    nbins=30,
    title="Age Distribution of Employees",
    labels={"age": "Age", "count": "Frequency"},
    color_discrete_sequence=["steelblue"],
)
fig.update_layout(bargap=0.05)
fig.show()

In [ ]:
# Gender distribution
gender_counts = employees["gender"].value_counts().reset_index()
gender_counts.columns = ["gender", "count"]

fig = px.pie(
    gender_counts,
    values="count",
    names="gender",
    title="Gender Distribution",
    color_discrete_sequence=px.colors.qualitative.Set2,
)
fig.show()

In [ ]:
# Education level distribution
edu_order = ["High School", "Bachelor", "Master", "PhD"]
edu_counts = employees["education_level"].value_counts().reindex(edu_order).reset_index()
edu_counts.columns = ["education_level", "count"]

fig = px.bar(
    edu_counts,
    x="education_level",
    y="count",
    title="Education Level Distribution",
    labels={"education_level": "Education Level", "count": "Number of Employees"},
    color_discrete_sequence=["teal"],
)
fig.show()

In [ ]:
# Tenure distribution (years at company)
fig = px.histogram(
    employees,
    x="tenure_years",
    nbins=25,
    title="Employee Tenure Distribution",
    labels={"tenure_years": "Tenure (Years)", "count": "Frequency"},
    color_discrete_sequence=["darkorange"],
)
fig.update_layout(bargap=0.05)
fig.show()

## 4. Performance Analysis

In [ ]:
# Performance score distribution
fig = px.histogram(
    evaluations,
    x="performance_score",
    nbins=40,
    title="Overall Performance Score Distribution",
    labels={"performance_score": "Performance Score", "count": "Frequency"},
    color_discrete_sequence=["mediumpurple"],
)
fig.update_layout(bargap=0.05)
fig.show()

# Summary statistics
evaluations["performance_score"].describe()

In [ ]:
# Performance by department (box plot)
# Merge evaluations with employee department info
eval_dept = evaluations.merge(
    employees[["employee_id", "department"]], on="employee_id", how="left"
)

fig = px.box(
    eval_dept,
    x="department",
    y="performance_score",
    color="department",
    title="Performance Score by Department",
    labels={"performance_score": "Performance Score", "department": "Department"},
)
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
# Performance trend over evaluation periods
period_avg = (
    evaluations.groupby("evaluation_period")["performance_score"]
    .mean()
    .reset_index()
    .sort_values("evaluation_period")
)

fig = px.line(
    period_avg,
    x="evaluation_period",
    y="performance_score",
    title="Average Performance Score Over Evaluation Periods",
    labels={
        "evaluation_period": "Evaluation Period",
        "performance_score": "Avg. Performance Score",
    },
    markers=True,
)
fig.show()

In [ ]:
# Performance vs Potential scatter (proto 9-box grid)
# Use the latest evaluation per employee
latest_eval = (
    evaluations.sort_values("evaluation_period")
    .groupby("employee_id")
    .last()
    .reset_index()
)
latest_eval = latest_eval.merge(
    employees[["employee_id", "department"]], on="employee_id", how="left"
)

fig = px.scatter(
    latest_eval,
    x="performance_score",
    y="potential_score",
    color="department",
    title="9-Box Grid: Performance vs Potential (Latest Evaluation)",
    labels={
        "performance_score": "Performance Score",
        "potential_score": "Potential Score",
    },
    opacity=0.6,
)

# Add quadrant reference lines at the 33rd and 66th percentiles
for pct in [0.33, 0.66]:
    perf_thresh = latest_eval["performance_score"].quantile(pct)
    pot_thresh = latest_eval["potential_score"].quantile(pct)
    fig.add_vline(x=perf_thresh, line_dash="dash", line_color="gray", opacity=0.5)
    fig.add_hline(y=pot_thresh, line_dash="dash", line_color="gray", opacity=0.5)

fig.show()

In [ ]:
# Correlation heatmap of evaluation scores
score_cols = [
    col for col in evaluations.columns
    if evaluations[col].dtype in ["float64", "int64"]
    and col != "employee_id"
]

corr_matrix = evaluations[score_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    square=True,
    linewidths=0.5,
    ax=ax,
)
ax.set_title("Correlation Heatmap of Evaluation Metrics", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Training Analysis

In [ ]:
# Training hours distribution
fig = px.histogram(
    training,
    x="training_hours",
    nbins=30,
    title="Distribution of Training Hours per Record",
    labels={"training_hours": "Training Hours", "count": "Frequency"},
    color_discrete_sequence=["coral"],
)
fig.update_layout(bargap=0.05)
fig.show()

In [ ]:
# Training completion rates by category
completion_rates = (
    training.groupby("training_category")["completed"]
    .mean()
    .reset_index()
    .rename(columns={"completed": "completion_rate"})
    .sort_values("completion_rate", ascending=False)
)
completion_rates["completion_rate"] = (completion_rates["completion_rate"] * 100).round(1)

fig = px.bar(
    completion_rates,
    x="training_category",
    y="completion_rate",
    title="Training Completion Rate by Category (%)",
    labels={
        "training_category": "Training Category",
        "completion_rate": "Completion Rate (%)",
    },
    color_discrete_sequence=["seagreen"],
    text="completion_rate",
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.show()

In [ ]:
# Training hours vs performance score
# Aggregate total training hours per employee
emp_training_hours = (
    training.groupby("employee_id")["training_hours"]
    .sum()
    .reset_index()
    .rename(columns={"training_hours": "total_training_hours"})
)

# Use average performance score per employee
emp_perf = (
    evaluations.groupby("employee_id")["performance_score"]
    .mean()
    .reset_index()
    .rename(columns={"performance_score": "avg_performance"})
)

train_perf = emp_training_hours.merge(emp_perf, on="employee_id", how="inner")
train_perf = train_perf.merge(
    employees[["employee_id", "department"]], on="employee_id", how="left"
)

fig = px.scatter(
    train_perf,
    x="total_training_hours",
    y="avg_performance",
    color="department",
    title="Total Training Hours vs Average Performance Score",
    labels={
        "total_training_hours": "Total Training Hours",
        "avg_performance": "Avg. Performance Score",
    },
    opacity=0.6,
    trendline="ols",
)
fig.show()

## 6. Promotions

In [ ]:
# Promotion rate by department
promoted_dept = promotions.merge(
    employees[["employee_id", "department"]], on="employee_id", how="left"
)
promo_rate = (
    promoted_dept.groupby("department").size().reset_index(name="promotions")
)
dept_size = employees.groupby("department").size().reset_index(name="total_employees")
promo_rate = promo_rate.merge(dept_size, on="department")
promo_rate["promotion_rate_pct"] = (
    (promo_rate["promotions"] / promo_rate["total_employees"]) * 100
).round(1)

fig = px.bar(
    promo_rate.sort_values("promotion_rate_pct", ascending=False),
    x="department",
    y="promotion_rate_pct",
    title="Promotion Rate by Department (%)",
    labels={"department": "Department", "promotion_rate_pct": "Promotion Rate (%)"},
    color_discrete_sequence=["royalblue"],
    text="promotion_rate_pct",
)
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.show()

In [ ]:
# Performance score at the time of promotion
fig = px.histogram(
    promotions,
    x="performance_at_promotion",
    nbins=25,
    title="Performance Score at Time of Promotion",
    labels={
        "performance_at_promotion": "Performance Score",
        "count": "Frequency",
    },
    color_discrete_sequence=["goldenrod"],
)
fig.update_layout(bargap=0.05)
fig.show()

print(f"Median performance at promotion: "
      f"{promotions['performance_at_promotion'].median():.2f}")

In [ ]:
# Salary increase distribution for promoted employees
fig = px.histogram(
    promotions,
    x="salary_increase_pct",
    nbins=25,
    title="Salary Increase (%) at Promotion",
    labels={"salary_increase_pct": "Salary Increase (%)", "count": "Frequency"},
    color_discrete_sequence=["indianred"],
)
fig.update_layout(bargap=0.05)
fig.show()

print(f"Average salary increase at promotion: "
      f"{promotions['salary_increase_pct'].mean():.1f}%")

## 7. Key Findings

Based on the exploratory analysis above, the following patterns emerge:

- **Performance distribution is roughly normal** with a slight positive skew, indicating most employees cluster around the median score with fewer extreme high or low performers.
- **Department-level variation is significant**: some departments consistently outperform others in average evaluation scores, suggesting differences in management quality, role difficulty, or evaluation calibration.
- **Training hours correlate positively with performance** — employees who invested more time in training programs tend to score higher on evaluations, though the relationship is moderate, indicating other factors also drive performance.
- **Promotion decisions appear merit-based**: promoted employees have notably higher performance scores than the general population, with a clear threshold effect visible in the distribution.
- **The 9-box grid reveals talent segments**: the performance-vs-potential scatter shows distinct clusters, with a small group of high-performance/high-potential employees forming the top-right quadrant — prime candidates for leadership development.
- **Training completion rates vary by category**: some training types see significantly lower completion, which may indicate scheduling issues, content relevance problems, or workload conflicts worth investigating.